# FaithGuard phase-1 detector pilot

**Run by:** M.L Ahamed

Which LettuceDetect encoder should Channel A start from? Fine-tunes both candidates (v1 ModernBERT-large and v2 mmBERT-base) in fp16 with gradient checkpointing at 1,024 and 2,048 tokens, finds the largest batch that fits, and checks that both released checkpoints run through the pinned lettucedetect package. Expect about 20-30 minutes.

**On Kaggle:** Create → New Notebook → File → Import Notebook → choose this file. In the right-hand panel set
*Accelerator* to **GPU T4 x2** and switch *Internet* **on** (needs a phone-verified account). Then *Run All*.

**On Colab:** File → Upload notebook. Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

**Afterwards:**
1. Download the `results` folder (Kaggle: *Output* panel; Colab: *Files* panel) and add the JSON files to `pilots/results/` in the repository.
2. Add a row to `logs/gpu-hours.csv`, using the **session hours the platform shows**, not just the script time.
3. Record the outcome in `docs/decision-log.md`.

In [ ]:
# Setup: pinned GPU packages (requirements/gpu.txt) and a look at the GPU.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import subprocess, sys
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS])
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

In [ ]:
%%writefile common.py
"""Shared helpers for the phase-1 GPU pilots.

Each pilot records the machine it ran on, times itself, and writes one JSON
result file plus a ready-to-paste row for logs/gpu-hours.csv.
"""

from __future__ import annotations

import datetime as dt
import importlib.metadata as md
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("FG_RESULTS_DIR", "results"))

PACKAGES = (
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
    "lettucedetect",
    "flash-linear-attention",
    "triton",
)


def detect_platform() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def package_version(name: str) -> str | None:
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


def nvidia_smi(query: str = "name,memory.total,memory.used,driver_version") -> list[str]:
    if not shutil.which("nvidia-smi"):
        return []
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    return [line.strip() for line in out.stdout.splitlines() if line.strip()]


def environment() -> dict:
    """Describe the machine, so every result can be traced to its hardware and versions."""
    info: dict = {
        "platform": detect_platform(),
        "python": sys.version.split()[0],
        "os": platform.platform(),
        "started_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "nvidia_smi": nvidia_smi(),
        "packages": {name: package_version(name) for name in PACKAGES},
        "gpus": [],
    }
    try:
        import torch
    except ImportError:
        return info
    info["torch_cuda"] = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            info["gpus"].append(
                {
                    "name": props.name,
                    "memory_gb": round(props.total_memory / 2**30, 2),
                    "compute_capability": f"{props.major}.{props.minor}",
                    # The T4 (7.5) and P100 (6.0) have no native bf16; that is why the pilots use fp16.
                    "native_bf16": props.major >= 8,
                }
            )
    return info


def gpu_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return "cpu"


class Stopwatch:
    def __enter__(self) -> "Stopwatch":
        self.start = time.perf_counter()
        self.seconds = 0.0
        return self

    def __exit__(self, *exc) -> None:
        self.seconds = time.perf_counter() - self.start


def write_result(name: str, result: dict) -> Path:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    path = RESULTS_DIR / f"{name}-{stamp}.json"
    path.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")
    print(f"\nResult written to {path}")
    return path


def print_gpu_log_row(task: str, script_hours: float, outcome: str, notes: str) -> None:
    """Print a row for logs/gpu-hours.csv.

    Kaggle and Colab charge for the whole session while a GPU is attached, so the
    member replaces script_hours with the session hours shown by the platform.
    """
    date = dt.date.today().isoformat()
    notes = notes.replace(",", ";")
    print("\nRow for logs/gpu-hours.csv (fill in member and account; use the session hours the platform shows):")
    print(f"{date},<member>,<account>,{detect_platform()},{gpu_name()},1,{task},{script_hours:.2f},{outcome},{notes}")


def pip_install(*requirements: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])


In [ ]:
%%writefile detector_pilot.py
"""Phase-1 detection pilot: which LettuceDetect encoder should Channel A start from?

Compares the two candidate checkpoints on a free T4:
    v1-large   KRLabsOrg/lettucedect-large-modernbert-en-v1   (ModernBERT-large)
    v2-mmbert  KRLabsOrg/lettucedect-v2-mmbert-base           (mmBERT-base, 0.3B)

For each checkpoint and sequence length (1,024 and 2,048 tokens) it fine-tunes in
fp16 mixed precision with gradient checkpointing, finds the largest batch that
fits, and records memory, speed and numeric stability. It then runs each released
checkpoint through the lettucedetect package on one financial example, to check
that the baseline and detector v0 work with the pinned package version.

Speed is only half of the choice: quality on a small development sample is
compared in phase 4. This pilot answers "does it fit, and how many GPU hours".

Examples:
    python detector_pilot.py
    python detector_pilot.py --models v2-mmbert --seq-lens 2048
    python detector_pilot.py --tiny        # CPU smoke test with tiny random encoders
"""

from __future__ import annotations

import argparse
import gc
import json
import math
import random
import statistics
import time

import torch

from common import Stopwatch, environment, print_gpu_log_row, write_result

MODELS = {
    "v1-large": "KRLabsOrg/lettucedect-large-modernbert-en-v1",
    "v2-mmbert": "KRLabsOrg/lettucedect-v2-mmbert-base",
}

EXAMPLE_CONTEXT = (
    "Income statement extract (Rs. '000). Profit after tax: Group 14,212,560 in 2025 and 13,004,180 in 2024; "
    "Bank 12,450,330 in 2025 and 11,635,900 in 2024."
)
EXAMPLE_QUESTION = "What was the Group's profit after tax in FY2025?"
EXAMPLE_ANSWER = "Group profit after tax rose 7.0% to Rs. 12,450 million."


def synthetic_prompt(rng: random.Random, n_rows: int) -> str:
    rows = []
    for i in range(n_rows):
        metric = rng.choice(["Revenue", "Profit after tax", "Total assets", "Finance costs", "Net interest income"])
        entity = rng.choice(["Group", "Company", "Bank"])
        rows.append(f"{metric} ({entity}) 2025: {rng.randint(10**5, 10**8):,}; 2024: {rng.randint(10**5, 10**8):,}.")
    return "Report extract (Rs. '000). " + " ".join(rows) + "\nQuestion: What was the Group's revenue in FY2025?"


def synthetic_answer(rng: random.Random) -> str:
    return (
        f"Group revenue rose {rng.randint(1, 30)}.{rng.randint(0, 9)}% to Rs. {rng.randint(1000, 90000):,} million "
        "in FY2025, while profit after tax was broadly stable."
    )


def make_batch(tokenizer, rng: random.Random, batch: int, seq_len: int) -> dict:
    """Prompt and answer encoded as a pair, as LettuceDetect does; only answer tokens are labelled.

    The prompt is longer than seq_len and truncated, so every sequence is exactly seq_len
    real tokens (worst case for memory). Answer tokens containing a digit get label 1:
    a trivial task, used only to check that the loss falls and stays finite.
    """
    prompts = [synthetic_prompt(rng, n_rows=seq_len // 8) for _ in range(batch)]
    answers = [synthetic_answer(rng) for _ in range(batch)]
    enc = tokenizer(prompts, answers, truncation="only_first", max_length=seq_len, padding="max_length", return_tensors="pt")
    labels = torch.full_like(enc["input_ids"], -100)
    for row in range(batch):
        seq_ids = enc.sequence_ids(row)
        for pos, seq in enumerate(seq_ids):
            if seq == 1:
                token = tokenizer.convert_ids_to_tokens(int(enc["input_ids"][row, pos]))
                labels[row, pos] = int(any(ch.isdigit() for ch in token))
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def tiny_model(model_id: str):
    from transformers import AutoConfig, AutoModelForTokenClassification

    config = AutoConfig.from_pretrained(model_id)
    config.hidden_size, config.intermediate_size = 64, 128
    config.num_hidden_layers, config.num_attention_heads = 2, 2
    return AutoModelForTokenClassification.from_config(config)


def train_steps(model, tokenizer, rng, device, batch, seq_len, steps, use_amp) -> dict:
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)
    scaler = torch.amp.GradScaler(device, enabled=use_amp)
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    losses, seconds, nonfinite, skipped = [], [], 0, 0
    for _ in range(steps):
        data = {k: v.to(device) for k, v in make_batch(tokenizer, rng, batch, seq_len).items()}
        start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device, dtype=torch.float16, enabled=use_amp):
            loss = model(**data).loss
        if not torch.isfinite(loss):
            nonfinite += 1
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scale_before = scaler.get_scale() if use_amp else 1.0
        scaler.step(optimizer)
        scaler.update()
        if use_amp and scaler.get_scale() < scale_before:
            skipped += 1
        if device == "cuda":
            torch.cuda.synchronize()
        seconds.append(time.perf_counter() - start)
        losses.append(loss.item())
    del optimizer
    warm = seconds[2:] if len(seconds) > 4 else seconds
    out = {
        "batch": batch,
        "losses": [round(x, 4) for x in losses],
        "nonfinite_losses": nonfinite,
        "skipped_steps": skipped,
        "seconds_per_step": round(statistics.mean(warm), 3),
        "tokens_per_second": round(batch * seq_len / statistics.mean(warm), 1),
    }
    if device == "cuda":
        out["peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    return out


def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_config(name, model_id, seq_len, args, device, rng) -> dict:
    from transformers import AutoModelForTokenClassification, AutoTokenizer

    use_amp = device == "cuda"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = tiny_model(model_id) if args.tiny else AutoModelForTokenClassification.from_pretrained(model_id)
    model.to(device)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()
    record = {"model": name, "model_id": model_id, "seq_len": seq_len, "params_m": round(model.num_parameters() / 1e6, 1)}

    largest = None
    for batch in args.batches:
        try:
            train_steps(model, tokenizer, rng, device, batch, seq_len, steps=2, use_amp=use_amp)
            largest = batch
        except torch.OutOfMemoryError:
            free_memory()
            break
    record["largest_batch"] = largest
    if largest is None:
        record["verdict"] = "FAIL"
        record["reasons"] = ["batch of 1 does not fit"]
    else:
        run = train_steps(model, tokenizer, rng, device, largest, seq_len, steps=args.steps, use_amp=use_amp)
        record.update(run)
        record["gpu_hours_per_million_tokens"] = round(1e6 / run["tokens_per_second"] / 3600, 2)
        head, tail = statistics.mean(run["losses"][:3]), statistics.mean(run["losses"][-3:])
        reasons = []
        if any(not math.isfinite(x) for x in run["losses"][len(run["losses"]) // 2 :]):
            reasons.append("non-finite loss in the second half")
        if run["skipped_steps"] > max(3, args.steps // 5):
            reasons.append(f"GradScaler skipped {run['skipped_steps']} steps (fp16 overflow)")
        if not tail < head:
            reasons.append("loss did not fall")
        record["verdict"] = "PASS" if not reasons else "FAIL"
        record["reasons"] = reasons
        if args.tiny:
            record["verdict"], record["reasons"] = "SMOKE-OK", ["tiny random model; pass criteria not applied"]
    del model
    free_memory()
    print(json.dumps({k: v for k, v in record.items() if k != "losses"}))
    return record


def inference_check(model_id: str) -> dict:
    """Run the released checkpoint through the lettucedetect package on one financial example."""
    from lettucedetect.models.inference import HallucinationDetector

    try:
        with Stopwatch() as load:
            detector = HallucinationDetector(method="transformer", model_path=model_id, max_length=4096)
        with Stopwatch() as pred:
            spans = detector.predict(
                context=[EXAMPLE_CONTEXT], question=EXAMPLE_QUESTION, answer=EXAMPLE_ANSWER, output_format="spans"
            )
        return {"ok": True, "load_seconds": round(load.seconds, 1), "predict_seconds": round(pred.seconds, 2), "spans": spans}
    except Exception as err:  # report, never crash the pilot on the optional check
        return {"ok": False, "error": f"{type(err).__name__}: {str(err)[:300]}"}
    finally:
        free_memory()


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--models", default="v1-large,v2-mmbert")
    parser.add_argument("--seq-lens", default="1024,2048")
    parser.add_argument("--batches", default="1,2,4,8,16", help="batch sizes tried in order until one runs out of memory")
    parser.add_argument("--steps", type=int, default=20)
    parser.add_argument("--seed", type=int, default=13)
    parser.add_argument("--skip-inference", action="store_true")
    parser.add_argument("--tiny", action="store_true", help="CPU smoke test with tiny random encoders")
    args = parser.parse_args()
    args.batches = [int(b) for b in args.batches.split(",")]
    seq_lens = [int(s) for s in args.seq_lens.split(",")]
    names = args.models.split(",")
    if args.tiny:
        seq_lens, args.batches, args.steps, args.skip_inference = [128], [1, 2], 4, True

    torch.manual_seed(args.seed)
    rng = random.Random(args.seed)
    device = "cuda" if torch.cuda.is_available() and not args.tiny else "cpu"
    if device == "cpu" and not args.tiny:
        raise SystemExit("No GPU found. Turn on a GPU accelerator, or use --tiny for a CPU smoke test.")

    started = time.perf_counter()
    result: dict = {"pilot": "detector", "settings": vars(args), "environment": environment(), "runs": [], "inference": {}}
    for name in names:
        for seq_len in seq_lens:
            result["runs"].append(run_config(name, MODELS[name], seq_len, args, device, rng))
        if not args.skip_inference:
            result["inference"][name] = inference_check(MODELS[name])
            print(name, json.dumps(result["inference"][name], default=str)[:400])
    script_seconds = time.perf_counter() - started
    result["script_seconds"] = round(script_seconds, 1)

    print("\nSUMMARY")
    for run in result["runs"]:
        print(
            f"{run['model']:10s} seq {run['seq_len']:5d}  {run['verdict']:8s} largest batch {run['largest_batch']}  "
            f"tokens/s {run.get('tokens_per_second', '-')}  GPU-h/1M tokens {run.get('gpu_hours_per_million_tokens', '-')}  "
            f"peak {run.get('peak_memory_gb', '-')} GB"
        )
    outcome = "PASS" if all(r["verdict"] in ("PASS", "SMOKE-OK") for r in result["runs"]) else "PARTIAL"
    write_result("detector" + ("-tiny" if args.tiny else ""), result)
    print_gpu_log_row(
        "detector pilot", script_seconds / 3600, outcome, f"models {' '.join(names)}; seq {' '.join(map(str, seq_lens))}"
    )


if __name__ == "__main__":
    main()


## Both encoders, both sequence lengths

In [ ]:
!python detector_pilot.py

## Summary of every result in this session

In [ ]:
import glob, json
for path in sorted(glob.glob("results/*.json")):
    r = json.load(open(path))
    print(f"\n== {path}")
    if r["pilot"] == "repair_lora":
        t = r.get("training", {})
        print(r["verdict"], r.get("reasons"), "| kernels:", r["kernels"]["gated_delta_rule"])
        print("tokens/s", t.get("tokens_per_second"), "| GPU-h per 1M tokens", t.get("gpu_hours_per_million_tokens"),
              "| peak GB", r.get("peak_memory_gb"), "| skipped steps", t.get("skipped_steps"))
    elif r["pilot"] == "detector":
        for run in r["runs"]:
            print(run["model"], run["seq_len"], run["verdict"], "| batch", run["largest_batch"],
                  "| tokens/s", run.get("tokens_per_second"), "| GPU-h per 1M tokens", run.get("gpu_hours_per_million_tokens"),
                  "| peak GB", run.get("peak_memory_gb"))
        for name, check in r["inference"].items():
            print("inference", name, "ok" if check.get("ok") else check.get("error"))
    elif r["pilot"] == "generation":
        for m in r["models"]:
            print(m["model"], m.get("error") or f"{m['best_answers_per_gpu_hour']} answers per GPU hour")
        print("GPU hours for all 920 answers:", r.get("gpu_hours_for_all_920_answers"))